In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/ml-notes

/content/drive/MyDrive/ml-notes


In [5]:
import os

In [6]:
os.environ["GITHUB_PAT"] = "github_pat_#"

In [7]:
!git add .

In [8]:
!git commit -m 'committing directly from colab'

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@77bcc008cdb4.(none)')


In [9]:
!git config --global user.email "pbainsla18082004@gmail.com"
!git config --global user.name "Pranjal Bainsla"

In [10]:
!git commit -m 'committing directly from colab'

[master 4536f23] committing directly from colab
 1 file changed, 0 insertions(+), 0 deletions(-)
 rename makemore_exercises_pt1.ipynb => exercises/makemore_exercises_pt1.ipynb (100%)


In [11]:
!git push


fatal: could not read Username for 'https://github.com': No such device or address


In [12]:
!git remote -v

origin	https://github.com/pranjalbainsla/ml-notes.git (fetch)
origin	https://github.com/pranjalbainsla/ml-notes.git (push)


In [13]:
!git remote set-url origin https://$GITHUB_PAT@github.com/pranjalbainsla/ml-notes.git

In [14]:
!git add .

In [15]:
!git commit -m 'committing directly from colab'

On branch master
Your branch is ahead of 'origin/master' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [16]:
!git push

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 369 bytes | 23.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/pranjalbainsla/ml-notes.git
   226b16f..4536f23  master -> master


In [17]:

!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

words = open("names.txt").read().splitlines()

--2026-07-05 09:15:49--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.04s   

2026-07-05 09:15:50 (5.14 MB/s) - ‘names.txt’ saved [228145/228145]



In [18]:
len(words)

32033

In [19]:
import torch
# N = torch.zeros((729, 27), dtype=torch.int32)
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [32]:
# create the dataset
xs, ys = [], []
for w in words:
  chs = ['.', '.'] + list(w) + ['.']
  for i in range(len(chs)-2):
    ix1a = stoi[chs[i]]
    ix1b = stoi[chs[i+1]]

    ix1 = 27 * ix1a + ix1b
    ix2 = stoi[chs[i+2]]

    xs.append(ix1)
    ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)
print(xs[:10])
print(ys[:10])

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((729, 27), generator=g, requires_grad=True)

number of examples:  228146
tensor([  0,   5, 148, 364, 352,   0,  15, 417, 333, 265])
tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9])


In [33]:
# gradient descent
for k in range(1):

  # forward pass
  xenc = torch.nn.functional.one_hot(xs, num_classes=729).float() # input to the network: one-hot encoding
  logits = xenc @ W # predict log-counts
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
  loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
  print(loss.item())

  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()

  # update
  W.data += -50 * W.grad

3.8028223514556885


In [34]:
# finally, sample from the 'neural net' model
g = torch.Generator().manual_seed(2147483647)

for i in range(5):

  out = []
  prev1 = 0
  prev2 = 0

  while True:

    # ----------
    # BEFORE:
    #p = P[ix]
    # ----------
    # NOW:
    ix = 27 * prev1 + prev2
    xenc = torch.nn.functional.one_hot(torch.tensor([ix]), num_classes=729).float()
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    # ----------

    next_char = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    if next_char == 0:
          break

    out.append(itos[next_char])
    prev1, prev2 = prev2, next_char

  print(''.join(out))

texzdfzjglkurxycczkwyhhmvlzimjtnagnrlkfdkzka
za
royxcpbbpwkhrggitmj
idzjedktvojkw
t


In [35]:
!git add .

In [36]:
!git commit -m 'trigram model using nn approach'

[master fb7f644] trigram model using nn approach
 1 file changed, 32033 insertions(+)
 create mode 100644 names.txt


In [37]:
!git push

Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 101.20 KiB | 2.81 MiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/pranjalbainsla/ml-notes.git
   4536f23..fb7f644  master -> master


In [42]:
!git add .


In [43]:
!git commit -m 'trigram nn approach'

[master 553e7a1] trigram nn approach
 1 file changed, 1 insertion(+)
 create mode 100644 exercises/makemore_ex_pt1_nn.ipynb


In [44]:
!git push

Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 4.65 KiB | 339.00 KiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/pranjalbainsla/ml-notes.git
   fb7f644..553e7a1  master -> master
